Reruns every E0 result, one numbered section per result, plus 0 for data creation.

## Setup
CPU unless marked GPU; rerun setup after switching runtime.
Teammates: in Drive, "Add shortcut to My Drive" for MJJS, e0_probe_cmp, e0_confirm, e0_injection (shared with you), so they sit at My Drive root.
Inputs: those original folders + MJJS/E0_probes_and_rollouts/data_copies. Outputs: MJJS/E0_probes_and_rollouts/rerun.
Until the PR merges into main, set BRANCH = 'siddhant/recovery-modal'.

In [ ]:
BRANCH = 'main'  # 'siddhant/recovery-modal' until the PR merges
REPO_DIR, REPO_URL = '/content/Ghait---MJJS', 'https://github.com/marksh3n/Ghait---MJJS.git'
import getpass, os, shutil, subprocess
tok = globals().get('tok') or getpass.getpass('GitHub PAT: ').strip()
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', REPO_URL.replace('https://', f'https://{tok}@'), REPO_DIR],
                       capture_output=True, text=True)
    print('clone', r.returncode, r.stderr[-300:].replace(tok, '***'))
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL])  # token stays out of .git/config

def checkout(branch):
    auth = REPO_URL.replace('https://', f'https://{tok}@')
    r = subprocess.run(f'git -C {REPO_DIR} fetch -q {auth} +refs/heads/{branch}:refs/remotes/origin/{branch} && '
                       f'git -C {REPO_DIR} checkout -q -f -B {branch} origin/{branch} && '
                       f'git -C {REPO_DIR} log -1 --format="%h %ad %s"', shell=True, capture_output=True, text=True)
    print(branch, '|', (r.stdout + r.stderr).strip().replace(tok, '***'))

checkout(BRANCH)
%cd /content/Ghait---MJJS

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ORIG = '/content/drive/MyDrive'  # e0_probe_cmp, e0_confirm, e0_injection: originals, read only
TEAM = f'{ORIG}/MJJS/E0_probes_and_rollouts'
COPIES = f'{TEAM}/data_copies'
ROOT = f'{TEAM}/rerun'  # reruns land here
OUT, CONF, INJ = f'{ROOT}/e0_probe_cmp', f'{ROOT}/e0_confirm', f'{ROOT}/e0_injection'
for d in (OUT, CONF, INJ):
    os.makedirs(d, exist_ok=True)
PREDS = f'{ORIG}/e0_probe_cmp/preds'  # section 2 writes a fresh copy to OUT/preds
for d in ('e0_probe_cmp', 'e0_confirm', 'e0_injection', 'MJJS'):
    print(d, 'ok' if os.path.isdir(f'{ORIG}/{d}') else 'MISSING: add shortcut to My Drive')

In [ ]:
# traces_B, the registered acts/meta pairs and the ceiling rollouts from HF
import hashlib
from huggingface_hub import HfApi, hf_hub_download, login
try:
    HfApi().whoami()
except Exception:
    login(token=getpass.getpass('HF token: '), add_to_git_credential=False)
SRC = 'MarkSh3n/e0-shared'
P32K = f'{COPIES}/p32k'  # 32k-position acts_B/meta_B; fallback: an HF copy that holds p32k/
P32K_REPO = f"{HfApi().whoami()['name']}/e0-shared"

def use_acts(which):
    # 'p32k': acts_B/meta_B at 12 positions to 32768 (Drive copy, else P32K_REPO). 'p6': 6 positions to 4096 (SRC root),
    # which only the main probe_cmp run (section 2) used
    for f in ['acts_B.npy', 'meta_B.parquet', 'meta_B.manifest.json']:
        if os.path.lexists(f'data/{f}'):
            os.remove(f'data/{f}')
        if which == 'p6':
            if not f.endswith('.json'):  # the root copy has no manifest
                shutil.copy(hf_hub_download(SRC, f, repo_type='dataset'), f'data/{f}')
        elif os.path.exists(f'{P32K}/{f}'):
            shutil.copy(f'{P32K}/{f}', f'data/{f}')
        else:
            shutil.copy(hf_hub_download(P32K_REPO, f'p32k/{f}', repo_type='dataset'), f'data/{f}')
    print('data/acts_B.npy is', which)

os.makedirs('data', exist_ok=True)
use_acts('p32k')
!python src/probe_cmp/fetch.py && python src/probe_cmp/validate.py | tail -3
ROLL = hf_hub_download(SRC, 'recovery_ceiling', repo_type='dataset')
print('traces_B sha256', hashlib.sha256(open('data/traces_B.jsonl', 'rb').read()).hexdigest()[:16], '(want c8cb3d8f9b21453b)')

In [ ]:
!python -c "import math_verify" 2>/dev/null || python -m pip install -q math-verify
!python -c "import datasets" 2>/dev/null || python -m pip install -q datasets
import json

In [ ]:
# once, from the account whose HF copy holds p32k/: put the 32k arrays in the team folder (~320 MB)
os.makedirs(P32K, exist_ok=True)
for f in ['acts_B.npy', 'meta_B.parquet', 'meta_B.manifest.json']:
    if not os.path.exists(f'{P32K}/{f}'):
        shutil.copy(hf_hub_download(P32K_REPO, f'p32k/{f}', repo_type='dataset'), f'{P32K}/{f}')
!ls -la {P32K}

## 0. Data creation
How every input was made. GPU (T4 enough). Rerun samples new traces; originals on HF `MarkSh3n/e0-shared` and Drive `e0_shared2`, use those for analysis.
Writes only to local `data/` and ROOT; nothing uploaded.

In [ ]:
import time
LOGS = '/content/logs'
os.makedirs(LOGS, exist_ok=True)

def need_vllm():
    # vllm 0.10.2 pins transformers<4.57, which keeps the second leading BOS
    if subprocess.run(['python', '-c', 'import vllm'], capture_output=True).returncode:
        subprocess.run('pip install -q uv && uv pip install --system -q vllm==0.10.2 "transformers<4.57" math-verify datasets',
                       shell=True, check=True)

def run_bg(name, cmd):
    subprocess.run(f'nohup {cmd} > {LOGS}/{name}.log 2>&1 &', shell=True, executable='/bin/bash')
    print(name, '| log', f'{LOGS}/{name}.log')

### 0a. Iteration 1: one trace per problem
367 MATH-500 L3–5 problems, 1 trace each, then steps-mode activations, probe, wrong-trace taxonomy. Outputs: `data/traces.jsonl`, `data/acts.npy`, `data/meta.parquet` (originals: `e0_shared2/acts.npy`, `meta.parquet`).

In [ ]:
need_vllm()
!python src/generate_vllm.py --max-new 32768  # flags as originally run
!python src/extract.py
!python src/probe.py
!python src/taxonomy.py

### 0b. Tag B: 367 problems × 6 traces
T 0.7, top_p 0.95, 32768 cap. Originals came in two blocks: pilot (lines 0–479, one BOS, vLLM 0.28.0) and main (480+, two BOS, CHUNK 32). This cell: both blocks on today's engine, two BOS everywhere. Output: `data/traces_B.jsonl` (original sha256 starts `c8cb3d8f9b21453b`).

In [ ]:
need_vllm()
!python src/generate_vllm.py --pilot --tag B --max-new 32768 --samples 6 --temp 0.7
!sed -i 's/^CHUNK = 8$/CHUNK = 32/' src/generate_vllm.py
run_bg('gen_B', 'python -u src/generate_vllm.py --tag B --max-new 32768 --samples 6 --temp 0.7')

### 0c. Tag B activations
Fixed positions 128…32768, 9 layers → `acts_B.npy` / `meta_B.parquet` (p32k arrays; 6-position arrays at HF root: same script, POSITIONS ending at 4096). Steps mode on mixed problems → `acts_BS.npy`, then last step only → `acts_BS_last.npy`.

In [ ]:
run_bg('ext_B', 'python -u src/extract.py --fixed --tag B')

In [ ]:
# mixed-outcome problems only, for steps-mode extraction
import json
ts = [json.loads(l) for l in open('data/traces_B.jsonl')]
by = {}
for t in ts:
    by.setdefault(t['problem_id'], []).append(t['trunc'] or not t['is_correct'])
mixed = {p for p, v in by.items() if 0 < sum(v) < len(v)}
with open('data/traces_BS.jsonl', 'w') as f:
    for t in ts:
        if t['problem_id'] in mixed:
            f.write(json.dumps(t) + '\n')
print(len(ts), 'traces,', len(mixed), 'mixed problems')
run_bg('ext_BS', 'python -u src/extract.py --tag BS')

In [ ]:
# after ext_BS finishes: keep the last step per trace (~30 MB instead of ~5 GB)
import numpy as np, pandas as pd
meta = pd.read_parquet('data/meta_BS.parquet')
last = meta[meta.step_idx == meta.n_steps - 1]
np.save('data/acts_BS_last.npy', np.asarray(np.load('data/acts_BS.npy', mmap_mode='r')[last.row.values]))
last.reset_index(drop=True).assign(row=range(len(last))).to_parquet('data/meta_BS_last.parquet')

### 0d. Recovery v2 pilot rollouts
20 problems, prefixes at fixed checkpoints, 8 continuations each, then top-up round. Originals: `e0_recovery_v2` / `e0_shared2/recovery_v2`. Output here: ROOT/e0_recovery_v2.

In [ ]:
need_vllm()
REC = f'{ROOT}/e0_recovery_v2'
os.makedirs(REC, exist_ok=True)
if not os.path.exists('data/recovery_v2'):
    os.symlink(REC, 'data/recovery_v2')
!python src/recovery/build_prefixes.py --tag B --n-problems 20 --seed 0
!python -u src/recovery/run_rollouts.py --temp 0.7 --smoke
run_bg('rec_v2', 'python -u src/recovery/run_rollouts.py --temp 0.7 --n 8')

In [ ]:
# after rec_v2: analysis, then 24 extra samples per prefix listed in topup.jsonl; rerun analysis after
!python src/recovery/analyze.py --tau 0.75
run_bg('rec_v2_r1', 'python -u src/recovery/run_rollouts.py --temp 0.7 --n-override-file data/recovery_v2/topup.jsonl --round 1')

### 0e. Ceiling rollouts (Modal, not Colab)
448 traces × cuts at 512 / 2048 / 4096 + t0, 8 continuations each → HF `recovery_ceiling`. Steps in `RUN_MODAL.md`; code `src/recovery/build_prefixes.py --design ceiling`, `run_rollouts.py`, `scripts/modal_app.py`.

## 1. Do hidden states predict correctness between problems?
Result: 0.887 vs 0.854, Δ +0.033 [−0.007, +0.074]. Output: e0_probe_cmp/summary.md (bottom section). Runtime: CPU, plus a 1.6 GB download.

In [ ]:
# iteration-1 meta and acts from the shared Drive (%run: Colab auth needs the kernel)
%run src/probe_cmp/fetch.py --iter1
!python src/probe_cmp/iter1.py --out {OUT}

## 2. Within problem, do probes beat length/tf-idf at fixed token counts?
Result: no checkpoint beats len (Holm p 1.00 before end); end 0.809 vs len 0.787, Δ +0.022 [−0.004, +0.048]. Output: e0_probe_cmp/summary.md, within_auroc_by_checkpoint.png, preds/. Runtime: CPU, ~35 min (re-extraction check: GPU).

In [ ]:
# GPU (T4) only: re-extracts 3 rows per source and names the token offset and layer mapping; summary.md reports it
!python src/probe_cmp/reextract_check.py --out {OUT} --n 3

In [ ]:
# this run read the 6-position arrays (11409 rows), so its population is 128..4096 + end
use_acts('p6')
!python -u src/probe_cmp/run.py --out {OUT} 2>&1 | tee -a {OUT}/full.log
!python src/probe_cmp/report.py --out {OUT}
use_acts('p32k')

## 3. Same, all 29 layers × 13 checkpoints
Result: 6144: 0.486 [0.41, 0.56]. Output: e0_probe_cmp/e0_full/ (e0_full_table.csv, run.log). Runtime: GPU (T4), hours for extraction, then ~1 h for the table.

In [ ]:
# extract_full.py (all layers, BOS matched to generation) and e0_full_table.py, not run.py.
# a 3-trace smoke test gates the full run, which goes to the background
E0F = f'{OUT}/e0_full'
os.makedirs(E0F, exist_ok=True)
!python -u scripts/colab/extract_full.py --traces data/traces_B.jsonl --out /content/e0_full_smoke --limit 3 && (nohup sh -c "python -u scripts/colab/extract_full.py --traces data/traces_B.jsonl --out /content/e0_full && python -u scripts/colab/e0_full_table.py --data /content/e0_full --traces data/traces_B.jsonl --rollouts {ROLL} --out {E0F} && cp /content/e0_full/acts_full.npy /content/e0_full/meta_full.parquet /content/e0_full/full_config.json {E0F}/ ; echo ALL DONE" > {E0F}/run.log 2>&1 &) && echo "full run launched, log at {E0F}/run.log"

In [ ]:
# progress; rerun any time
!tail -n 3 {E0F}/run.log; ls /content/e0_full/parts 2>/dev/null | wc -l

## 4. Does a window/representation change help?
Result: no view beats chance after Holm (best 0.572 [0.514, 0.629] at 512, Holm p 0.20); all trail r̂. Output: e0_probe_cmp/reps/ (reps_summary.md), e0_probe_cmp/reps_L16/. Runtime: GPU (T4) for the extraction, then CPU.

In [ ]:
# GPU (T4): last token and trailing-128 mean at each cut, layers 20 and 16, ceiling population -> reps/reps.npz
!python -u src/probe_cmp/extract_reps.py --rollouts {ROLL} --traces data/traces_B.jsonl --out {OUT}/reps --layers 20 16

In [ ]:
!python -u src/probe_cmp/reps.py --reps {OUT}/reps/reps.npz --rollouts {ROLL} --traces data/traces_B.jsonl --preds {PREDS} --out {OUT}/reps
!python -u src/probe_cmp/reps.py --reps {OUT}/reps/reps.npz --rollouts {ROLL} --traces data/traces_B.jsonl --preds {PREDS} --layer 16 --out {OUT}/reps_L16

## 5. Where did the 6144 tree come from?
Result: hgb/topk50 at 6144 picked as best cell. Output: e0_probe_cmp/p6144.log, p8192.log, L20-16-12/. Runtime: CPU, ~25 + ~30 min.

In [ ]:
# checkpoint sets and extra layers chosen after the main run; outputs go to OUT/L20-16-12/subset_*
!python -u src/probe_cmp/run.py --out {OUT} --checkpoints 128,256,512,1024,2048,4096,6144,8192,end --layers 20 16 12 --exploratory 2>&1 | tee -a {OUT}/p8192.log
!python -u src/probe_cmp/run.py --out {OUT} --checkpoints 128,256,512,1024,2048,4096,6144,end --layers 20 16 12 --exploratory 2>&1 | tee -a {OUT}/p6144.log
!python src/probe_cmp/report.py --out {OUT}/L20-16-12/subset_128_256_512_1024_2048_4096_6144_end

## 6. Does it reproduce, and on which cohort?
Result: 342/74: tree 0.523 vs len 0.554; 227/59: 0.573 vs 0.568; no rule gives the memo's 197/53. Output: e0_confirm/provenance_6144/, provenance_6144.log. Runtime: CPU, ~20–30 min.

In [ ]:
# the two 6144 prediction exports from the jaden/confirmation-planning branch, to compare trace sets
os.makedirs(CONF, exist_ok=True)
auth = REPO_URL.replace('https://', f'https://{tok}@')
subprocess.run(f'git -C {REPO_DIR} fetch -q {auth} +refs/heads/jaden/confirmation-planning:refs/remotes/origin/jaden/confirmation-planning',
               shell=True, capture_output=True)
for n in ['pop342', 'cohort227']:
    subprocess.run(f'git -C {REPO_DIR} show origin/jaden/confirmation-planning:experiments/jaden_confirmation/data/'
                   f'development/e0_6144_preds_{n}.csv > /content/{n}.csv', shell=True)
!wc -l /content/pop342.csv /content/cohort227.csv
!python -u src/confirm/provenance_6144.py --data data --out {CONF}/provenance_6144 \
    --exports /content/pop342.csv /content/cohort227.csv 2>&1 | tee {CONF}/provenance_6144.log

## 7. Frozen monitor fit (tree + probes + baselines)
Result: tree@6144 0.560 [0.487, 0.631]; len 0.554 [0.475, 0.630]. Output: e0_confirm/monitors/, fit_v2.log (fit.log and monitors_20261025/ are the earlier fit). Runtime: CPU, ~5 min.

In [ ]:
!python -u src/confirm/fit_monitors.py --data data --out {CONF}/monitors 2>&1 | tee {CONF}/fit_v2.log
!sha256sum {CONF}/monitors/manifest.json {CONF}/monitors/*.joblib

## 8. Is the tree signal pre-answer? (S7)
Result: +0.090 [−0.029, +0.211], 61 mixed problems, estimate-only. Output: e0_confirm/dev_dryrun_v2/tests.json. Runtime: CPU, ~10 min.

In [ ]:
# the confirmation analysis on dev data (dry run); S7 is one of its tests. section 12 reads the same output
!python -u src/confirm/dev_bundle.py --data data --rollouts {ROLL} --monitors {CONF}/monitors \
    --out {CONF}/dev_bundle_v2 2>&1 | tail -25
!python -u src/confirm/analyze.py --bundle {CONF}/dev_bundle_v2 --out {CONF}/dev_dryrun_v2 2>&1 | tee {CONF}/dryrun_v2.log | tail -150

## 9. Does the prefix carry WP outcome information?
Result: r̂ 0.626 / 0.655 / 0.811; minus best probe at 4096 +0.262 [+0.183, +0.338]; 0.777 at 4096 before the answer value appears. Output: e0_probe_cmp/ceiling/ (ceiling_summary.md, recovery_curves.png). Runtime: CPU.

In [ ]:
!python src/probe_cmp/ceiling.py --rollouts {ROLL} --traces data/traces_B.jsonl --meta data/meta_B.parquet --preds {PREDS} --out {OUT}/ceiling

## 10. How much could rollout-ranked intervention buy?
Result: at 4096, q 0.25: +5.7 pp [+4.1, +7.2] over none at 42,886 tokens/attempt vs 11,964 (dry run, incl. selection rollouts); carries the sampler gap. Output: e0_confirm/oracle_v4/{all,main,pilot}/. Runtime: CPU, ~10 min.

In [ ]:
!python src/probe_cmp/oracle.py --rollouts {ROLL} --traces data/traces_B.jsonl \
    --preds {PREDS} --problems all main pilot --out {CONF}/oracle_v4 2>&1 | tail -60

## 11. Earlier oracle versions
Result: superseded by v4. Output: e0_probe_cmp/oracle/, oracle_v2/, oracle_v3/, oracle_lenmatched/. Runtime: none; earlier versions of oracle.py, rerun section 10 instead (the length-matched run is in section 17).

## 12. Does a monitor improve accuracy at equal tokens?
Result: +1.1 pp [−0.6, +3.1] vs random; +1.6 [−0.1, +3.4] vs maj@3; S1–S6 Holm p 1.00. Output: e0_confirm/dev_dryrun_v2/report.md, dryrun_v2.log (dryrun.log is the earlier run). Runtime: run in section 8.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{CONF}/dev_dryrun_v2/report.md').read()))

## 13. Power of the planned confirmation
Result: detect power 0.18–0.51 shared-source; rule-out of ≥ 5 pp 0.99. Output: e0_confirm/discordance.json, discordance.log. Runtime: CPU, ~1 min.

In [ ]:
# effect and discordance of the primary policy on the dry-run bundle, then McNemar / Tango power (power.py) at N = 1000
!python -u src/confirm/discordance.py --bundle {CONF}/dev_bundle_v2 --traces data/traces_B.jsonl \
    --out {CONF}/discordance.json 2>&1 | tee {CONF}/discordance.log

## 14. Latest confirmation-pipeline outputs
Result: —. Output: e0_confirm/ (summary_20261027.txt collects the logs of sections 7, 8 and 13 and these hashes). Runtime: CPU, ~5 min.

In [ ]:
# counts only: live cuts and mixed problems kept by each pre-answer rule; no monitor scores
!python -u src/confirm/count_pre_answer.py --data data --rollouts {ROLL} 2>&1 | tee {CONF}/pre_answer_counts.log
# the manifest hash goes into docs/CONFIRMATION_SPEC.md
!sha256sum {CONF}/monitors/manifest.json {CONF}/monitors/*.joblib

## 15. Sampling temperature of the traces
Result: T̂ ≈ 0.70 in every block. Output: e0_probe_cmp/temp_blocks.json. Runtime: GPU (T4), ~20–40 min.

In [ ]:
# per-block likelihood MLE, teacher-forced with src/recovery/check_temp.py's scoring
!python -u scripts/colab/temp_blocks.py --traces data/traces_B.jsonl --out {OUT}/temp_blocks.json

## 16. BOS in source traces
Result: lines 0–479 one BOS, 480+ two. Output: e0_probe_cmp/ceiling/bos_check.csv. Runtime: GPU (T4), a few minutes.

In [ ]:
# reads the original lenfilter_prefixes.csv (section 17 writes a fresh one to OUT)
!python -u scripts/colab/bos_check.py --prefixes {ORIG}/e0_probe_cmp/ceiling/lenfilter_prefixes.csv --traces data/traces_B.jsonl --out {OUT}/ceiling/bos_check.csv

## 17. Why are rollouts more accurate than sources?
Result: gap 7.3 / 5.1 / 3.0 pp at 512 / 2048 / 4096; unexplained. Output: e0_probe_cmp/ceiling/ (lenfilter_prefixes.csv, mixed_sel_problems.csv), e0_probe_cmp/oracle_lenmatched/. Runtime: CPU.

In [ ]:
# length-filtered r_hat per prefix, then source accuracy vs what mixed-problem selection predicts
!python scripts/colab/lenfilter.py --rollouts {ROLL} --traces data/traces_B.jsonl --meta data/meta_B.parquet --preds {PREDS} --out {OUT}/ceiling/lenfilter_prefixes.csv
!python scripts/colab/mixed_sel.py --prefixes {OUT}/ceiling/lenfilter_prefixes.csv --traces data/traces_B.jsonl --out {OUT}/ceiling/mixed_sel_problems.csv

In [ ]:
# selection residual split by generation block; a problem's traces never span blocks
import pandas as pd
line = {json.loads(l)['trace_id']: i for i, l in enumerate(open('data/traces_B.jsonl'))}
prob = pd.read_csv(f'{OUT}/ceiling/mixed_sel_problems.csv', index_col=0)
pref = pd.read_csv(f'{OUT}/ceiling/lenfilter_prefixes.csv')
pop = pref.dropna(subset=['trace_id']).drop_duplicates('trace_id')
blk = pop.groupby('problem_id').trace_id.agg(lambda ts: {line[t] < 480 for t in ts})
assert (blk.map(len) == 1).all()
prob['block'] = blk.map(lambda s: 'pilot, lines 0-479 (one BOS)' if s.pop() else 'main, lines 480+ (two BOS)')
w = lambda g, c: (g[c] * g.n_eligible).sum() / g.n_eligible.sum()
rows = [{'block': k, 'problems': len(g), 'traces': int(g.n_eligible.sum()), 'source acc': w(g, 'source_acc'),
         'pred': w(g, 'pred'), 'obs - pred': w(g, 'source_acc') - w(g, 'pred'), 'mean p_hat': w(g, 'p_hat')}
        for k, g in list(prob.groupby('block')) + [('all', prob)]]
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
# restart / backtrack oracle with continuations held to the source traces' length rule
!python src/probe_cmp/oracle.py --rollouts {ROLL} --traces data/traces_B.jsonl --preds {PREDS} --length-matched --out {OUT}/oracle_lenmatched

## 18. Do natural arithmetic slips separate wrong traces?
Result: 17.6 vs 15.7 slips per 1k claims (wrong vs correct), no CI. Output: e0_injection/slip_summary_B.json, audit_B_round2_labeled.csv. Runtime: CPU.

In [ ]:
!python src/injection/slip_scan.py --tag B --audit 40
!mkdir -p {INJ} && cp data/injection/* {INJ}/

In [ ]:
# precision of the hand-labelled round-2 audit
import pandas as pd
a = pd.read_csv(f'{INJ}/audit_B_round2_labeled.csv')
a['lab'] = a['real (y/n)'].astype(str).str.strip().str.lower().str[0]
ok = a[a.lab.isin(['y', 'n'])]
print(f"labeled {len(ok)}/{len(a)}, unsure {(a.lab == '?').sum()}, precision {(ok.lab == 'y').mean():.2f}")
for col in ['trace_label', 'slip_type', 'context']:
    g = ok.groupby(col).lab
    print(pd.DataFrame({'n': g.size(), 'precision': g.apply(lambda s: (s == 'y').mean()).round(2)}), '\n')

## 19. Does the model register an injected digit error?
Result: not run. Output: e0_injection/ (sites_B.jsonl, scores_B.jsonl, inject_summary_B.json). Runtime: GPU (T4), not yet run.

In [ ]:
# matched benign vs consequential digit edits of correct claims, teacher-forced; reads claims_B.jsonl from section 18.
# the 2-pair smoke must print an identity line with kl_max 0.0 before the full run
!python -u src/injection/inject.py select --tag B --out-dir {INJ}
!python -u src/injection/inject.py score --tag B --out-dir {INJ} --limit 2

In [ ]:
# resumable: rerun after a disconnect
!python -u src/injection/inject.py score --tag B --out-dir {INJ}
!python -u src/injection/inject.py report --tag B --out-dir {INJ}

## Superseded
Kept in `E0_colab_archive` only: recover.py v1 pilot (20 traces × 4), `probe_fixed.py` 6-position probe, inline within-problem baselines, EB refit. All replaced by sections 2, 9, 17.